# Module 3: The Full Stack Agent

## LLM + MCP + Tool Calling

In this module, you'll learn how to:
- Connect to an MCP server using Streamable HTTP transport
- Discover available tools from the MCP server
- Integrate LLM inference with MCP tool calling
- Build an agentic chat loop that can use external tools

---

## Step 1: Install Dependencies

We need additional packages for MCP client functionality.

In [ ]:
!pip install -r requirements.txt

## Step 2: Import Libraries

We'll use:
- `openai` - For LLM inference
- `mcp` - Model Context Protocol client
- `httpx` - HTTP client for API calls
- `asyncio` - For async operations (MCP uses async)

In [ ]:
import os
import json
import asyncio
from typing import Optional
from contextlib import AsyncExitStack

import httpx
from openai import OpenAI
from dotenv import load_dotenv

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

# Load environment variables
load_dotenv()

print("Libraries loaded successfully!")

## Step 3: Configure Environment

Make sure your `.env` file contains:

```env
OPENAI_BASE_URL=<your-api-endpoint>
OPENAI_API_KEY=<your-api-key>
MCP_SERVER_URL=<your-mcp-server-url>
MCP_AUTH_TOKEN=<your-mcp-auth-token>
```

In [ ]:
# Verify environment variables are set
required_vars = ["OPENAI_BASE_URL", "OPENAI_API_KEY", "MCP_SERVER_URL", "MCP_AUTH_TOKEN"]

for var in required_vars:
    value = os.getenv(var)
    if value:
        # Mask sensitive values
        masked = value[:10] + "..." if len(value) > 10 else value
        print(f"{var}: {masked}")
    else:
        print(f"{var}: NOT SET (required!)")

## Step 4: Create HTTP Client Factory

We need a custom HTTP client factory for MCP that handles SSL verification (useful for self-signed certificates in development).

In [ ]:
def create_insecure_httpx_client(**kwargs):
    """Create an httpx client with SSL verification disabled for self-signed certificates."""
    return httpx.AsyncClient(verify=False, **kwargs)

print("HTTP client factory defined!")

## Step 5: Build the MCP Client Class

This is the core of our agent - it combines:
1. **OpenAI client** for LLM inference
2. **MCP session** for tool discovery and execution
3. **Tool calling loop** to handle LLM requests for tools

In [ ]:
class MCPClient:
    """MCP Client for interacting with an MCP Streamable HTTP server"""

    def __init__(self, model_name: str = "gpt-oss-120b"):
        self.model_name = model_name
        self.session: Optional[ClientSession] = None
        self.exit_stack = AsyncExitStack()
        self._session_context = None
        self._streams_context = None
        
        # Initialize OpenAI client
        http_client = httpx.Client(verify=False)
        self.openai = OpenAI(
            base_url=os.getenv("OPENAI_BASE_URL"),
            api_key=os.getenv("OPENAI_API_KEY"),
            http_client=http_client
        )
        print(f"OpenAI client initialized (model: {model_name})")

    async def connect_to_server(self, server_url: str, headers: Optional[dict] = None):
        """Connect to an MCP server running with HTTP Streamable transport"""
        print(f"\nConnecting to MCP server at: {server_url}")
        
        # Create streamable HTTP connection
        self._streams_context = streamablehttp_client(
            url=server_url,
            headers=headers or {},
            httpx_client_factory=create_insecure_httpx_client,
            timeout=60.0
        )
        read_stream, write_stream, _ = await self._streams_context.__aenter__()
        print("Streams established successfully")

        # Create MCP session
        self._session_context = ClientSession(read_stream, write_stream)
        self.session = await self._session_context.__aenter__()
        print("Session context created")

        # Initialize the session
        print("Initializing session...")
        await self.session.initialize()
        print("Session initialized successfully!")

    async def list_available_tools(self):
        """List all tools available from the MCP server"""
        response = await self.session.list_tools()
        return response.tools

    async def cleanup(self):
        """Properly clean up the session and streams"""
        if self._session_context:
            await self._session_context.__aexit__(None, None, None)
        if self._streams_context:
            await self._streams_context.__aexit__(None, None, None)
        await self.exit_stack.aclose()
        print("Cleanup completed")

print("MCPClient class defined!")

## Step 6: Connect to the MCP Server

Let's establish a connection to our MCP server.

In [ ]:
# Create client instance
client = MCPClient(model_name="nai-gpt-oss-120b")

# Connect to MCP server
await client.connect_to_server(
    server_url=os.getenv("MCP_SERVER_URL"),
    headers={
        "Authorization": f"Bearer {os.getenv('MCP_AUTH_TOKEN')}"
    }
)

## Step 7: Discover Available Tools

Let's see what tools the MCP server provides. These will be available for our LLM to call.

In [ ]:
# List available tools
tools = await client.list_available_tools()

print(f"Found {len(tools)} tools:\n")
for tool in tools:
    print(f"  - {tool.name}")
    print(f"    Description: {tool.description}")
    print()

## Step 8: Convert MCP Tools to OpenAI Format

The OpenAI API expects tools in a specific format. Let's create a helper function to convert MCP tools.

In [ ]:
def mcp_tools_to_openai_format(mcp_tools):
    """Convert MCP tools to OpenAI function calling format"""
    return [
        {
            "type": "function",
            "function": {
                "name": tool.name,
                "description": tool.description,
                "parameters": tool.inputSchema,
            }
        }
        for tool in mcp_tools
    ]

# Convert our tools
openai_tools = mcp_tools_to_openai_format(tools)
print(f"Converted {len(openai_tools)} tools to OpenAI format")
print("\nExample tool structure:")
print(json.dumps(openai_tools[0], indent=2) if openai_tools else "No tools available")

## Step 9: Implement Query Processing with Tool Calling

This is where the magic happens! We:
1. Send the user query to the LLM with available tools
2. If the LLM wants to call a tool, execute it via MCP
3. Send the tool result back to the LLM for final response

In [ ]:
async def process_query(client: MCPClient, query: str) -> str:
    """Process a query using LLM and available MCP tools"""
    
    # Start with user message
    messages = [{"role": "user", "content": query}]
    
    # Get available tools from MCP server
    mcp_tools = await client.list_available_tools()
    openai_tools = mcp_tools_to_openai_format(mcp_tools)
    
    # Call LLM with tools
    response = client.openai.chat.completions.create(
        model=client.model_name,
        max_tokens=1000,
        messages=messages,
        tools=openai_tools if openai_tools else None,
    )
    
    message = response.choices[0].message
    final_text = []
    
    # Check if LLM wants to call tools
    if message.tool_calls:
        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)
            
            print(f"  [Tool Call] {tool_name}")
            print(f"  [Arguments] {tool_args}")
            
            # Execute tool via MCP
            result = await client.session.call_tool(tool_name, tool_args)
            
            print(f"  [Result] {str(result.content)[:100]}...")
            final_text.append(f"[Called {tool_name}]")
            
            # Add assistant message and tool result to conversation
            messages.append({"role": "assistant", "content": message.content or ""})
            messages.append({"role": "user", "content": str(result.content)})
            
            # Get final response from LLM
            response = client.openai.chat.completions.create(
                model=client.model_name,
                max_tokens=1000,
                messages=messages,
            )
            
            final_text.append(response.choices[0].message.content)
    else:
        # No tool calls, just return the response
        final_text.append(message.content)
    
    return "\n".join(final_text)

print("Query processing function defined!")

## Step 10: Test a Single Query

Let's test our agent with a query that might trigger tool usage.

In [ ]:
# Test with a sample query
test_query = "What tools do you have available?"

print(f"Query: {test_query}\n")
response = await process_query(client, test_query)
print(f"\nResponse:\n{response}")

## Step 11: Build the Interactive Chat Loop

Now let's create a full interactive chat that can use tools!

In [ ]:
async def interactive_chat(client: MCPClient):
    """Run an interactive chat loop with tool calling support"""
    
    print("="*60)
    print("MCP Agent Chat Started!")
    print("Your AI assistant can now use external tools.")
    print("Type 'quit' to exit, 'tools' to list available tools")
    print("="*60)
    print()
    
    while True:
        try:
            query = input("You: ").strip()
            
            if query.lower() == 'quit':
                print("Goodbye!")
                break
            elif query.lower() == 'tools':
                tools = await client.list_available_tools()
                print("\nAvailable tools:")
                for tool in tools:
                    print(f"  - {tool.name}: {tool.description}")
                print()
                continue
            elif not query:
                continue
            
            print()
            response = await process_query(client, query)
            print(f"\nAssistant: {response}\n")
            
        except KeyboardInterrupt:
            print("\nGoodbye!")
            break
        except Exception as e:
            import traceback
            print(f"\nError: {str(e)}")
            traceback.print_exc()
            print()

print("Interactive chat function defined!")
print("Run the next cell to start chatting with your agent.")

In [ ]:
# Start the interactive chat!
await interactive_chat(client)

## Step 12: Cleanup

Always clean up your MCP connections when done.

In [ ]:
# Clean up the client connection
await client.cleanup()

---

## Summary

In this module, you learned:

1. **MCP Connection** - Connect to an MCP server using Streamable HTTP transport
2. **Tool Discovery** - List and inspect available tools from the server
3. **Format Conversion** - Convert MCP tools to OpenAI function calling format
4. **Tool Execution** - Execute tools via MCP and handle results
5. **Agentic Loop** - Build a chat that seamlessly uses external tools

---

## Architecture

```
+--------+     +---------+     +------------+     +-------------+
|  User  | --> |   LLM   | --> | Tool Call? | --> | MCP Server  |
+--------+     +---------+     +------------+     +-------------+
    ^                               |                    |
    |                               v                    v
    +<-------- Final Response <---- + <-- Tool Result <--+
```

---

## Next Steps

Move to **Module 4: Code Whisperer** where you'll build an autonomous code review agent!

```bash
cd ../module-04
```